# SFH posterior from S-PLUS colours (part 3)

A normalizing flow (trained by `train_ml_model.py` on the 3090) gives each galaxy a posterior over the parameters of a delayed-τ star-formation history with one metallicity: log age, log τ, log metallicity, the dust (`av_ism`, `log_tau_bc`), `logU`, the formed and the surviving mass, and the redshift. Every other label (`age`, `tau`, `mw_age`, `f_young`, `f_mid`, `f_old`, `log_sfr`, `log_ssfr`, ...) is derived from the samples with the same analytic functions as the training labels. Inputs are the 11 colours against r, the r magnitude (only for the absolute masses), and z_obs with its uncertainty (0 for spectroscopic redshifts).

**Inputs:** `ml_runs/mlp/`, `synthetic_validation.parquet`, `synthetic_test.parquet`, `training_set.parquet` (prior draws), `spectroscopic_reference.csv`, `dr3_corrected.csv`.
**Outputs:** `ml_evaluation_summary.csv`, `ml_sfh_catalogue.parquet`.
**Run:** top to bottom with the *Inference4ETGs (MultiNest)* kernel. Sections marked **LOCKED** run only with `RUN_LOCKED_TEST = True`, once, after the user has inspected the validation results: they use the synthetic test set and the reference set's test half.

Posterior samples for the same seed differ between CUDA and CPU (`DEVICE`): numbers are reproducible per device, not across devices. `ml_networks.load_run` refuses a checkpoint trained for other targets (e.g. the 16-bin runs): retrain it.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import torch

import ml_data
import ml_evaluation as ev
import ml_networks
import synthetic_model as sm

RUN_LOCKED_TEST = False            # set True once, for the final report
RUN_DIRECTORIES = {"mlp": "ml_runs/mlp"}
CHOSEN = "mlp"                     # the CNN lost in the 16-bin comparison, so only the MLP is trained
VALIDATION_FILE = "synthetic_validation.parquet"
TEST_FILE = "synthetic_test.parquet"
TRAINING_FILE = "training_set.parquet"
REFERENCE_FILE = "spectroscopic_reference.csv"
SPLUS_FILE = "dr3_corrected.csv"
SUMMARY_FILE = "ml_evaluation_summary.csv"
CATALOGUE_FILE = "ml_sfh_catalogue.parquet"

N_POSTERIOR_SAMPLES = 1000         # per galaxy for evaluation
N_CATALOGUE_SAMPLES = 500          # per galaxy for the DR3 catalogue
SAMPLING_SEED = 7
VALIDATION_SEEDS = {"spectroscopic": 100, "photometric": 101}   # same copies as training used
TEST_SEEDS = {"spectroscopic": 200, "photometric": 201}
# The 9 flow targets, then the derived labels that describe the SFH
KEY_LABELS = [*ml_data.TARGET_NAMES, "mw_age", "f_young", "f_mid", "f_old", "log_sfr"]
# Labels compared with the Bagpipes posteriors: the four judged ones, and mw_age which is only reported
JUDGED_LABELS = sm.CALIBRATION_LABELS
REFERENCE_LABELS = sm.CALIBRATION_LABELS + ["mw_age"]
PRIOR_WINDOW_DZ = 0.05             # prior draws within this redshift of a galaxy define its prior width
MIN_PROB_GALAXY = 0.8
PASSIVE_SSFR_LIMIT = -11            # [log10 yr^-1] spectral sSFR below this = passive galaxy
PRIOR_SEED = 11                     # brightness offsets of the prior draws
N_JOBS = 11
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # flow sampling is autoregressive: slow on CPU

runs = {}
for name, directory in RUN_DIRECTORIES.items():
    runs[name] = ml_networks.load_run(directory, device=DEVICE)

## 1. Training and validation

The MLP is the only model (the CNN lost in the 16-bin comparison). This section shows its training history and the coverage of the key labels on the validation set.

In [ ]:
GALAXY_CHUNK = 500                 # galaxies decoded at once: all 16k x 1000 samples x 18 labels would need ~2.4 GB


def labelled_posteriors(run, observations, n_samples, seed, labels=None):
    """Decoded posterior labels for every valid galaxy: dict label -> array (n_valid, n_samples).

    labels: names to keep (default all 18 sm.TRAINING_LABELS). Decoding runs in galaxy chunks, so only the kept labels
    are ever held for the whole set.
    """
    estimator, _, standardisers = run
    valid = observations["valid"]
    if not valid.any():
        raise ValueError("no valid galaxy (every galaxy lacks a detected r band or a brightness donor): "
                         "nothing to predict")
    flat = observations["flat"][valid]
    banded = observations["banded"][valid]
    n_galaxies = len(flat)

    posteriors = {}
    for chunk_start in range(0, n_galaxies, GALAXY_CHUNK):
        chunk = slice(chunk_start, chunk_start + GALAXY_CHUNK)
        theta = ml_networks.posterior_samples(estimator, standardisers, flat[chunk], banded[chunk],
                                              n_samples, seed + chunk_start, DEVICE)
        n_chunk = theta.shape[0]
        decoded = ml_data.decode_samples(theta.reshape(-1, ml_data.N_TARGETS))
        for column in decoded.columns:
            if labels is not None and column not in labels:
                continue
            if column not in posteriors:
                posteriors[column] = np.empty((n_galaxies, n_samples))
            posteriors[column][chunk] = decoded[column].to_numpy().reshape(n_chunk, n_samples)
    truths = ml_data.decode_samples(observations["targets"][valid])
    return posteriors, truths


validation_table = pd.read_parquet(VALIDATION_FILE)
figure, axis = plt.subplots(figsize=(6, 4))
choice_rows = []
for name, run in runs.items():
    history = pd.read_csv(f"{RUN_DIRECTORIES[name]}/{ml_networks.HISTORY_FILE}")
    axis.plot(history["epoch"], history["validation_loss"], label=f"{name} validation")
    axis.plot(history["epoch"], history["train_loss"], ls=":", label=f"{name} train")
    for mode, seed in VALIDATION_SEEDS.items():
        observations = ml_data.validation_observations(validation_table, mode, np.random.default_rng(seed))
        posteriors, truths = labelled_posteriors(run, observations, N_POSTERIOR_SAMPLES, SAMPLING_SEED, KEY_LABELS)
        for label in KEY_LABELS:
            coverage = ev.coverage_curve(posteriors[label], truths[label].to_numpy(), [0.68, 0.95])
            choice_rows.append({"embedding": name, "mode": mode, "label": label,
                                "coverage_68": coverage[0], "coverage_95": coverage[1],
                                "best_validation_loss": history["validation_loss"].min()})
axis.set(xlabel="epoch", ylabel="mean −log p", title="training history")
axis.legend()
choice = pd.DataFrame(choice_rows)
choice["coverage_error"] = np.abs(choice["coverage_68"] - 0.68) + np.abs(choice["coverage_95"] - 0.95)
by_embedding = choice.groupby("embedding").agg(best_validation_loss=("best_validation_loss", "first"),
                                               mean_coverage_error=("coverage_error", "mean"))
display(by_embedding.round(4))
print(f"Model: {CHOSEN}")
plt.show()

## 2. Calibration, accuracy and information per parameter (LOCKED: synthetic test set)

For each redshift mode, the left panel is the coverage curve of the key labels (a calibrated posterior lies on the diagonal) and the right panel the histogram of the rank of the truth among the posterior samples (flat means calibrated, U-shaped means too narrow). The table gives coverage, bias, NMAD and posterior/prior width for every key label: the 9 flow targets and the derived SFH labels.

Several derived labels have point masses in the prior (`f_young` and `f_old` are often exactly 0 or 1, `log_sfr` sits on its floor when nothing formed in the last 100 Myr). Ties make the inclusive coverage too high and pile the ranks at one edge, and a zero prior width gives a NaN width ratio; read those rows as report-only.

In [ ]:
def mass_rescaled_prior(prior_table, catalogue, seed):
    """Prior draws sorted by redshift, with the mass-like labels at observed brightness.

    Why: training_set.parquet holds models at a fixed mass; an observed galaxy has the mass of its
    brightness (sm.scale_to_observed_r, as in training), so the prior width of log_mstar is ~1 dex, not 0.08.
    Inputs: prior_table: training_set.parquet; catalogue: DataFrame with r_auto and zml of DR3 galaxies;
            seed: for the brightness donors. Returns: DataFrame without rows lacking a donor, sorted by redshift.
    """
    offsets = sm.scale_to_observed_r(prior_table[sm.BANDS].to_numpy(dtype=float),
                                     prior_table["redshift"].to_numpy(dtype=float),
                                     catalogue["r_auto"].to_numpy(), catalogue["zml"].to_numpy(),
                                     np.random.default_rng(seed))[1]
    rescaled = sm.rescale_mass_labels(prior_table, offsets)
    return rescaled[np.isfinite(offsets)].sort_values("redshift").reset_index(drop=True)


def windowed_width_ratio(posterior, prior_table, label, true_redshift):
    """Posterior/prior width of a label, the prior taken within PRIOR_WINDOW_DZ of each galaxy's redshift."""
    prior_widths = ev.windowed_prior_widths(prior_table["redshift"].to_numpy(), prior_table[label].to_numpy(),
                                            true_redshift, PRIOR_WINDOW_DZ)
    return ev.width_ratio(posterior, None, prior_widths=prior_widths)


if RUN_LOCKED_TEST:
    test_table = pd.read_parquet(TEST_FILE)
    catalogue = pd.read_csv(SPLUS_FILE, usecols=["PROB_GAL", "r_auto", "zml"])
    catalogue = catalogue[catalogue["PROB_GAL"] > MIN_PROB_GALAXY]
    prior_table = mass_rescaled_prior(pd.read_parquet(TRAINING_FILE), catalogue, PRIOR_SEED)
    summary_rows = []
    for mode, seed in TEST_SEEDS.items():
        observations = ml_data.validation_observations(test_table, mode, np.random.default_rng(seed))
        posteriors, truths = labelled_posteriors(runs[CHOSEN], observations, N_POSTERIOR_SAMPLES, SAMPLING_SEED,
                                                 KEY_LABELS)
        figure, axes = plt.subplots(1, 2, figsize=(12, 5))
        for label in KEY_LABELS:
            truth = truths[label].to_numpy()
            coverage = ev.coverage_curve(posteriors[label], truth)
            bias, nmad = ev.bias_and_nmad(np.nanmedian(posteriors[label], axis=1), truth)
            ratio = windowed_width_ratio(posteriors[label], prior_table, label, truths["redshift"].to_numpy())
            summary_rows.append({"section": "synthetic_test", "mode": mode, "label": label,
                                 "coverage_68": ev.coverage_curve(posteriors[label], truth, [0.68])[0],
                                 "coverage_95": ev.coverage_curve(posteriors[label], truth, [0.95])[0],
                                 "bias": bias, "nmad": nmad, "width_ratio": ratio})
            axes[0].plot(ev.NOMINAL_LEVELS, coverage, marker="o", label=label)
            axes[1].hist(ev.truth_ranks(posteriors[label], truth), bins=20, histtype="step", label=label)
        axes[0].plot([0, 1], [0, 1], color="black", lw=1)
        axes[0].set(xlabel="nominal coverage", ylabel="empirical coverage", title=f"{mode} redshift")
        axes[1].set(xlabel="rank of truth", ylabel="galaxies")
        axes[0].legend(fontsize=6, ncol=2)
        plt.show()
    summary = pd.DataFrame(summary_rows)
    display(summary.round(3))
else:
    print("synthetic test evaluation: skipped (locked)")

In [ ]:
if RUN_LOCKED_TEST:
    # Information per parameter: posterior width / prior width of each flow target
    # (1 = the colours say nothing about that parameter)
    BAR_WIDTH = 0.4
    figure, axis = plt.subplots(figsize=(9, 4))
    for mode_index, mode in enumerate(TEST_SEEDS):
        mode_rows = summary[(summary["mode"] == mode) & summary["label"].isin(ml_data.TARGET_NAMES)]
        width_ratios = mode_rows.set_index("label").loc[ml_data.TARGET_NAMES, "width_ratio"]
        axis.bar(np.arange(ml_data.N_TARGETS) + (mode_index - 0.5) * BAR_WIDTH, width_ratios,
                 width=BAR_WIDTH, label=f"{mode} redshift")
    axis.axhline(1, color="grey", ls=":")
    axis.set_xticks(range(ml_data.N_TARGETS))
    axis.set_xticklabels(ml_data.TARGET_NAMES, rotation=45, ha="right")
    axis.set(ylabel="posterior / prior 68% width", title="information per parameter (1 = prior only)")
    axis.legend()
    plt.show()

    # Accuracy against brightness, spectroscopic mode (the loop above ended on the photometric posteriors)
    spectroscopic_rng = np.random.default_rng(TEST_SEEDS["spectroscopic"])
    observations = ml_data.validation_observations(test_table, "spectroscopic", spectroscopic_rng)
    posteriors, truths = labelled_posteriors(runs[CHOSEN], observations, N_POSTERIOR_SAMPLES, SAMPLING_SEED,
                                             ["mw_age", "log_mstar"])
    r_magnitude = observations["flat"][observations["valid"], ml_data.SCALAR_INPUT_SLICE.start]
    r_quartile = pd.qcut(r_magnitude, 4, labels=["brightest", "second", "third", "faintest"])
    accuracy_rows = []
    for label in ("mw_age", "log_mstar"):
        medians = np.nanmedian(posteriors[label], axis=1)
        for quartile in r_quartile.categories:
            in_quartile = np.asarray(r_quartile == quartile)
            bias, nmad = ev.bias_and_nmad(medians[in_quartile], truths[label].to_numpy()[in_quartile])
            accuracy_rows.append({"label": label, "r_quartile": quartile, "bias": bias, "nmad": nmad})
    display(pd.DataFrame(accuracy_rows).round(3))
else:
    print("information per parameter and accuracy against brightness: skipped (locked)")


## 3. Agreement with the Bagpipes spectral posteriors

ML posteriors from the 3″ aperture colours and spectroscopic redshift, against each reference galaxy's spectral posterior. Development uses the `validation` half; the `test` half is LOCKED.

In [5]:
PHOTOMETRIC_PREFIX = "phot_"     # REFERENCE_FILE columns of the photometric Bagpipes fits (notebook 2, 6b)
reference = pd.read_csv(REFERENCE_FILE, dtype={"specobjid": str})
halves = ["validation", "test"] if RUN_LOCKED_TEST else ["validation"]
aperture_columns = []
aperture_error_columns = []
for band in sm.BANDS:
    aperture_columns.append(f"{band}_aper")
    aperture_error_columns.append(f"e_{band}_aper")

# Why one sampling pass per half: the spectral and photometric comparisons (cell below) must use the same
# ML draws of each galaxy, so the ML posterior is drawn once for every galaxy with either reference fit
has_spectral_fit = reference["fit_status"] == "fitted"
has_photometric_fit = pd.Series(False, index=reference.index)
if f"{PHOTOMETRIC_PREFIX}mw_age_median" in reference.columns:
    has_photometric_fit = reference[f"{PHOTOMETRIC_PREFIX}mw_age_median"].notna()
ml_posteriors_by_half = {}       # half -> (galaxies with an ML posterior, dict label -> (n, samples))
agreement_rows = []
for half in halves:
    galaxies = reference[(reference["split"] == half) & (has_spectral_fit | has_photometric_fit)]
    galaxies = galaxies.reset_index(drop=True)
    if len(galaxies) == 0:
        print(f"{half}: no fitted galaxies yet")
        continue
    observations = ml_data.colour_inputs(galaxies[aperture_columns].to_numpy(),
                                         galaxies[aperture_error_columns].to_numpy(),
                                         galaxies["z"].to_numpy(), np.zeros(len(galaxies)))
    if not observations["valid"].any():
        print(f"{half}: no galaxy with a detected r band in the aperture photometry")
        continue
    observations["targets"] = np.zeros((len(galaxies), ml_data.N_TARGETS))   # unknown; not used
    posteriors, _ = labelled_posteriors(runs[CHOSEN], observations, N_POSTERIOR_SAMPLES, SAMPLING_SEED)
    kept = galaxies[observations["valid"]].reset_index(drop=True)
    ml_posteriors_by_half[half] = (kept, posteriors)

    # The spectral comparison: galaxies with a spectral fit only
    spectral = (kept["fit_status"] == "fitted").to_numpy()
    spectral_galaxies = kept[spectral].reset_index(drop=True)
    spectral_posteriors = {label: posteriors[label][spectral] for label in REFERENCE_LABELS}
    galaxy_details = spectral_galaxies.set_index("specobjid")
    for row in ev.reference_comparison_rows(spectral_posteriors, spectral_galaxies, REFERENCE_LABELS):
        details = galaxy_details.loc[row["specobjid"]]
        is_star_forming = details["log_ssfr_median"] > PASSIVE_SSFR_LIMIT
        agreement_rows.append({"half": half, **row,
                               "galaxy_type": "star-forming" if is_star_forming else "passive",
                               "fibre": details["fiber_diameter"],
                               "calibrated_fitter": details["fitter_calibrated"]})
agreement = pd.DataFrame(agreement_rows)
if len(agreement) > 0:
    display(agreement.groupby(["half", "label"]).agg(median_offset=("offset_in_widths", "median"),
                                                      inside_68=("inside_68", "mean"),
                                                      inside_95=("inside_95", "mean"),
                                                      galaxies=("specobjid", "count")).round(3))
    display(agreement.groupby(["half", "label", "galaxy_type"])["inside_95"].mean().unstack().round(3))

median_offset  inside_68  inside_95  galaxies
half       label                                                   
validation Z_fin             -4.324      0.103      0.471        87
           f_old              1.295      0.391      0.529        87
           log_mstar          1.648      0.149      0.414        87
           mw_age             3.133      0.218      0.299        87

galaxy_type           passive  star-forming
half       label                           
validation Z_fin        0.456         0.500
           f_old        0.544         0.500
           log_mstar    0.386         0.467
           mw_age       0.228         0.433

### 3b. The "works" test: photometric and spectral Bagpipes

The ML is compared with two Bagpipes references, each of them a different question:

- **Photometric Bagpipes (emulator check).** The photometric fits (notebook 2, section 6b) fit the same 3″ photometry the ML sees, under the training prior and the simulator's likelihood, so a faithful emulator must agree with them.
- **Spectral Bagpipes (physical check).** The spectral fits use the SDSS spectra; agreement tells whether the colours recover what the spectra say.

A label agrees with a reference set when at least `ev.MIN_INSIDE_95` (80%) of the reference medians lie inside the ML 95% interval and the median offset is within ±`ev.MAX_ABS_MEDIAN_OFFSET` combined 1σ widths. The judged labels are `log_age`, `log_tau`, `log_metallicity` and `log_mstar` (`sm.CALIBRATION_LABELS`: age, τ and Z in log because the prior spans several decades of each); `mw_age` is reported, not judged. Only the `validation` half is used. Both verdicts are printed side by side; the notebook draws no conclusion beyond them.

In [ ]:
REFERENCE_SET_NAMES = {"photometric": "photometric Bagpipes (emulator check)",
                       "spectral": "spectral Bagpipes (physical check)"}
FITTER_CALIBRATION_COLUMNS = {"photometric": f"{PHOTOMETRIC_PREFIX}fitter_calibrated",
                              "spectral": "fitter_calibrated"}
REPORTED_ONLY_LABELS = ["mw_age"]

# 1. The validation galaxies with a converged photometric fit, with the ML draws of cell 8 (the test half stays
# locked)
n_validation = int((reference["split"] == "validation").sum())
photometric_agreement_rows = []
n_photometric_fits = 0
n_photometric_not_converged = 0
if "validation" in ml_posteriors_by_half:
    validation_galaxies, validation_posteriors = ml_posteriors_by_half["validation"]
    photometric = validation_galaxies[f"{PHOTOMETRIC_PREFIX}mw_age_median"].notna().to_numpy()
    n_photometric_fits = int(photometric.sum())
    if n_photometric_fits > 0:
        converged = validation_galaxies[f"{PHOTOMETRIC_PREFIX}converged"].astype("boolean").fillna(False)
        n_photometric_not_converged = int((photometric & ~converged.to_numpy(dtype=bool)).sum())
        compared = photometric & converged.to_numpy(dtype=bool)
        photometric_galaxies = validation_galaxies[compared].reset_index(drop=True)
        photometric_posteriors = {label: validation_posteriors[label][compared] for label in REFERENCE_LABELS}
        photometric_agreement_rows = ev.reference_comparison_rows(photometric_posteriors, photometric_galaxies,
                                                                  REFERENCE_LABELS, prefix=PHOTOMETRIC_PREFIX)
n_photometric_compared = len({row["specobjid"] for row in photometric_agreement_rows})
print(f"Photometric fits: {n_photometric_compared} of {n_validation} validation galaxies compared "
      f"({n_photometric_fits} with a fit and an ML posterior, {n_photometric_not_converged} not converged "
      "and left out)")
if n_photometric_fits == 0:
    print(f"No photometric Bagpipes fits in {REFERENCE_FILE} yet: run notebook 2 after the photometric fits")

# 2. The same verdict for both reference sets, validation half only. The "all" row covers the judged labels;
# the reported-only labels follow it
spectral_validation_rows = agreement
if len(agreement) > 0:
    spectral_validation_rows = agreement[agreement["half"] == "validation"]
comparison_rows = {"photometric": photometric_agreement_rows, "spectral": spectral_validation_rows}
verdicts = {}
for reference_set, rows in comparison_rows.items():
    judged_verdict = ev.agreement_verdict(rows, JUDGED_LABELS)
    reported_verdict = ev.agreement_verdict(rows, REPORTED_ONLY_LABELS).iloc[:-1]   # drop its "all" row
    verdicts[reference_set] = pd.concat([judged_verdict, reported_verdict], ignore_index=True)

# 3. Both verdicts side by side. Why the fitter flag: a reference whose fitter failed its calibration test
# (or whose calibration mocks are not all fitted) is a weaker reference, so it is shown next to the verdict
for reference_set, verdict in verdicts.items():
    fitter_flags = pd.Series(dtype=object)
    if FITTER_CALIBRATION_COLUMNS[reference_set] in reference.columns:
        fitter_flags = reference[FITTER_CALIBRATION_COLUMNS[reference_set]].dropna()
    fitter_calibrated = "unknown (calibration mocks not all fitted)"
    if len(fitter_flags) > 0:
        fitter_calibrated = fitter_flags.iloc[0]
    print(f"Agreement with {REFERENCE_SET_NAMES[reference_set]} (validation half; "
          f"{', '.join(REPORTED_ONLY_LABELS)} reported, not judged); fitter calibrated: {fitter_calibrated}")
    display(verdict.round(3))
    judged_rows = verdict[verdict["label"] == ev.ALL_LABELS_ROW]
    print(f"  the ML agrees on {', '.join(JUDGED_LABELS)}: {bool(judged_rows['agrees'].iloc[0])}")


## 4. S-PLUS DR3 catalogue

Photometric-z posteriors for every DR3 galaxy that passes the coverage test; the others are flagged.

In [ ]:
splus = pd.read_csv(SPLUS_FILE)
splus = splus[splus["PROB_GAL"] > MIN_PROB_GALAXY].reset_index(drop=True)
magnitudes = splus[[f"{band}_auto" for band in sm.BANDS]].to_numpy()
errors = splus[[f"e_{band}_auto" for band in sm.BANDS]].to_numpy()
detected = sm.detected_bands(magnitudes, errors)
zml = splus["zml"].to_numpy()
sigma_z = sm.SIGMA_Z * (1 + zml)

# 1. Coverage test (colour-based through its free offset), in parallel blocks
training_models = pd.read_parquet(TRAINING_FILE, columns=["redshift", *sm.BANDS])
blocks = np.array_split(np.arange(len(splus)), 4 * N_JOBS)
tasks = []
for block in blocks:
    tasks.append(delayed(sm.coverage_test)(magnitudes[block], errors[block], detected[block], zml[block],
                                           sm.redshift_window(zml[block]), training_models[sm.BANDS].to_numpy(),
                                           training_models["redshift"].to_numpy()))
coverage = pd.concat(Parallel(n_jobs=N_JOBS)(tasks), ignore_index=True)

# 2. Posterior summaries (16th, 50th, 84th percentile of every decoded label, the delayed-tau ones among
# them) for covered galaxies with a detected r band
inputs = ml_data.colour_inputs(magnitudes, errors, zml, sigma_z)
predictable = inputs["valid"] & coverage["covered"].to_numpy()
estimator, _, standardisers = runs[CHOSEN]
CATALOGUE_CHUNK = 2000             # galaxies per chunk: all samples at once would need ~4 GB
SUMMARY_PERCENTILES = ((16, "p16"), (50, "median"), (84, "p84"))
predictable_indices = np.flatnonzero(predictable)
summaries = {}                     # column name -> values for every DR3 galaxy
start = time.time()
for chunk_start in range(0, len(predictable_indices), CATALOGUE_CHUNK):
    chunk = predictable_indices[chunk_start:chunk_start + CATALOGUE_CHUNK]
    theta = ml_networks.posterior_samples(estimator, standardisers, inputs["flat"][chunk], inputs["banded"][chunk],
                                          N_CATALOGUE_SAMPLES, SAMPLING_SEED + chunk_start, DEVICE)
    decoded = ml_data.decode_samples(theta.reshape(-1, ml_data.N_TARGETS))
    for column in decoded.columns:
        values = decoded[column].to_numpy().reshape(len(chunk), N_CATALOGUE_SAMPLES)
        for percentile, suffix in SUMMARY_PERCENTILES:
            name = f"{column}_{suffix}"
            if name not in summaries:
                summaries[name] = np.full(len(splus), np.nan)
            summaries[name][chunk] = np.nanpercentile(values, percentile, axis=1)
seconds = time.time() - start

catalogue = splus[["RA", "DEC", "zml"]].copy()
catalogue["no_r_band"] = ~inputs["valid"]
catalogue["out_of_domain"] = inputs["valid"] & ~coverage["covered"].to_numpy()
# Why one concat: adding ~54 columns one by one fragments the frame (a pandas PerformanceWarning)
catalogue = pd.concat([catalogue, pd.DataFrame(summaries, index=catalogue.index)], axis=1)
n_predicted = len(predictable_indices)
catalogue.to_parquet(CATALOGUE_FILE, index=False)
print(f"{n_predicted} of {len(splus)} galaxies predicted ({catalogue['out_of_domain'].mean():.1%} out of domain, "
      f"{catalogue['no_r_band'].sum()} without r); {n_predicted / seconds:.0f} galaxies/s on {DEVICE}")

In [8]:
summary_tables = [summary] if RUN_LOCKED_TEST else []
summary_tables.append(choice.assign(section="validation_choice"))
for reference_set, verdict in verdicts.items():
    summary_tables.append(verdict.assign(section=f"agreement_{reference_set}"))
pd.concat(summary_tables, ignore_index=True).to_csv(SUMMARY_FILE, index=False)
print(f"wrote {SUMMARY_FILE}")

wrote ml_evaluation_summary.csv
